In [132]:
from math import gcd
from functools import reduce
import os
from collections import defaultdict
from itertools import permutations
from contextlib import ExitStack
from enum import Enum

class Transformation(Enum):
    #the class that defines the transformations that we apply to our series
    P = "P"
    I = "I"
    R = "R"
    RI = "RI"
    def apply(self, series:list[int], t:int, n:int)->list[int]:
        if   self == Transformation.P:  return [(x+t) % n for x in series]
        elif self == Transformation.I:  return [(-x+t) % n for x in series]
        elif self == Transformation.R:  return [(x+t) % n for x in reversed(series)]
        elif self == Transformation.RI: return [(-x+t) % n for x in reversed(series)]

class NotesConverter():
    def __init__(self):
        self.noteToInt = {"C":0, "C#":1, "Db":1,
                          "D":2, "D#":3, "Eb":3,
                          "E":4,
                          "F":5, "F#":6, "Gb":6,
                          "G":7, "G#":8, "Ab":8,
                          "A":9, "A#":10, "Bb":10,
                          "B":11}

    def convert(self, notes):
        try:
            values = [self.noteToInt[n] for n in notes]
        except KeyError as e:
              raise ValueError(f"Invalid note: {e}")
        base = values[0]
        normalized = [(x-base) % 12 for x in values]
        mapping = {v:i for i,v in enumerate(sorted(set(normalized)))}
        result = [mapping[x] for x in normalized]
        reverseMapping = {result[i]:notes[i] for i in range(len(notes))}
        return result, reverseMapping


def proliferatingPermutation(transformation:Transformation, series:list[int], t:int, n:int)->list[int]:
    #calculates the permutations obtained from series by appliyng transformation and transposition t.
    #The permutation is stored as a list[int] so that permutations[i] yields the note in the transformed
    #series that is in the same place that i was in the original series
    transformed = transformation.apply(series, t, n)
    index = {value: i for (i, value) in enumerate(series)}
    return [transformed[index[value]] for value in range(n)]

def lcm_list(series:list[int])->int:
    return reduce(lambda a,b: a*b//gcd(a,b), series)

def cycleStructure(permutation:list[int],n:int)->(int,list[int]):
    #calculates the lengths of the cycles of the permutation. Returns the order of the permutation and
    #a list with the lengths of the cycles
    visited = [False]*n
    cycles = []
    for i in range(n):
        if not visited[i]:
            current=permutation[i]
            length=1
            while current!=i:
                visited[current]=True
                current=permutation[current]
                length+=1
            cycles.append(length)
    return lcm_list(cycles), sorted(cycles)

def print_justified(series):
    width = max(len(str(x)) for x in series)
    print("[" + ", ".join(f"{str(x):<{width}}" for x in series) + "]")

def proliferations(transformation:Transformation, series:list[int], t:int,flag=False)->None:
    #first major program: takes a transformation with a transposition t and a series (and translates
    #it from notes to numbers if necessary). Prints the original and all its proliferations.
    #Also prints the order and the structure of the permutation obtained
    if isinstance(series[0],str):
        flag = True
        converter = NotesConverter()
        series,reverseMapping = converter.convert(series)
    else:
        reverseMapping = {i:i for i in series}
    print("Proliferations of the series using", transformation.name,"and a transposition of", t, "tone-fractions:")
    print_justified([reverseMapping[i] for i in series])
    n=len(series)
    permutation = proliferatingPermutation(transformation, series, t, n)
    current = [permutation[x] for x in series]
    while current != series:
        print_justified([reverseMapping[i] for i in current])
        current = [permutation[x] for x in current]
    (order,cycles)=cycleStructure(permutation,n)
    print("Order:", order)
    print("Structure:", cycles)
    if flag:
        print("-------------")
        print("Identification of notes and numbers:")
        width = max(len(str(k)) for k,v in reverseMapping.items())
        for k,v in reverseMapping.items():
            print(f"{k:<{width}} <---> {v}")

def proliferations_data(transformation:Transformation, t:int, n:int, path:str)->None:
    #second major program: takes a transformation, a transposition t, a number of notes n and a path.
    #The program creates in this path a folder for the transformation, another folder inside for the
    #number of notes and 3 more subfolders inside it, called "CompleteList", "Data_Orders" and
    #"Data_Structures". Finally, a txt file inside each of them distinguishing the transposition t.
    #The file in "CompleteList" shows every possible series starting with 0 and the number of proliferations
    #that it produces (a file with a very large size, you may not want it). The file in "Data_Orders" shows how many series generate permutations of each possible
    #order, and the file in "Data_Structures" shows how many series generate permutations of each possible
    #structure.
    base_path = os.path.join(path, transformation.name, f"Proliferations_{n}_notes")
    dirs = {
        "Data_Structures": os.path.join(base_path, "Data_Structures"),
        #"CompleteList": os.path.join(base_path, "CompleteList"),   #<--------- uncomment if you want the CompleteList
        "Data_Orders": os.path.join(base_path, "Data_Orders")
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    file_paths = {key: os.path.join(dirs[key], f"transposition{t}.txt") for key in dirs}
    occurrence_orders = defaultdict(int)
    occurrence_structures = defaultdict(int)
    with ExitStack() as stack:
        files = {key: stack.enter_context(open(file_paths[key], "w")) for key in file_paths}
        for seriesWithout0 in permutations(range(1, n)):
            completeSeries = (0,) + (seriesWithout0)
            permutation = proliferatingPermutation(transformation, completeSeries, t, n)
            order, structure = cycleStructure(permutation, n)
            #files["CompleteList"].write(f"{completeSeries} --> {order}\n")  #<--- uncomment if you want the CompleteList with the proliferations of each possible series
            occurrence_orders[order] += n #we sum n rather than 1 since we are only considering series that start with 0, so each will appear n times more
            occurrence_structures[(order, tuple(structure))] += n
        for k, v in sorted(occurrence_orders.items()):
            files["Data_Orders"].write(f"{k}: {v}\n")
        for k, v in sorted(occurrence_structures.items()):
            files["Data_Structures"].write(f"{k}: {v}\n")

In [133]:
#Examples used in the article:

# Stockhausen in freundschtaft
series1 = ["C", "F", "E", "C#", "B", "Bb", "Eb", "Ab", "D", "G", "A", "F#"]

# Op. 31 schoenberg
series2 = ["Bb", "E", "F#", "D#", "F", "A", "D", "C#", "G", "G#", "B", "C"]

# Op. 24 Webern
series3 = ["B", "Bb", "D", "Eb", "G", "F#", "G#", "E", "F", "C", "C#", "A"]

# Denisov sonata
series4 = ["B", "C", "A", "G#", "A#", "F", "F#", "G", "E", "D#", "C#", "D"]

# Symphony Webern
series5 = ["A", "F#", "G", "Ab", "E", "F", "B", "Bb", "D", "C#", "C", "Eb"]

# Acento en rosa
series6 = ["G", "C", "F#", "G#", "D#", "E", "A", "C#", "D", "F", "B", "A#"]

# Au dela du hazard
series7 = ["C", "G#", "G", "C#", "E", "D", "A#", "D#", "B", "F", "F#", "A"]


In [145]:
proliferations(Transformation.I, ["G", "C", "F#", "G#", "D#", "E", "A", "C#", "D", "F", "B", "A#"], 2)

Proliferations of the series using I and a transposition of 2 tone-fractions:
[G , C , F#, G#, D#, E , A , C#, D , F , B , A#]
[A , E , A#, G#, C#, C , G , D#, D , B , F , F#]
Order: 2
Structure: [1, 1, 2, 2, 2, 2, 2]
-------------
Identification of notes and numbers:
0  <---> G
5  <---> C
11 <---> F#
1  <---> G#
8  <---> D#
9  <---> E
2  <---> A
6  <---> C#
7  <---> D
10 <---> F
4  <---> B
3  <---> A#


In [135]:
#You can also input the list with numbers
proliferations(Transformation.P,[0, 5, 4, 1, 11, 10, 3, 8, 2, 7, 9, 6],1)

Proliferations of the series using P and a transposition of 1 tone-fractions:
[0 , 5 , 4 , 1 , 11, 10, 3 , 8 , 2 , 7 , 9 , 6 ]
[1 , 6 , 5 , 2 , 0 , 11, 4 , 9 , 3 , 8 , 10, 7 ]
[2 , 7 , 6 , 3 , 1 , 0 , 5 , 10, 4 , 9 , 11, 8 ]
[3 , 8 , 7 , 4 , 2 , 1 , 6 , 11, 5 , 10, 0 , 9 ]
[4 , 9 , 8 , 5 , 3 , 2 , 7 , 0 , 6 , 11, 1 , 10]
[5 , 10, 9 , 6 , 4 , 3 , 8 , 1 , 7 , 0 , 2 , 11]
[6 , 11, 10, 7 , 5 , 4 , 9 , 2 , 8 , 1 , 3 , 0 ]
[7 , 0 , 11, 8 , 6 , 5 , 10, 3 , 9 , 2 , 4 , 1 ]
[8 , 1 , 0 , 9 , 7 , 6 , 11, 4 , 10, 3 , 5 , 2 ]
[9 , 2 , 1 , 10, 8 , 7 , 0 , 5 , 11, 4 , 6 , 3 ]
[10, 3 , 2 , 11, 9 , 8 , 1 , 6 , 0 , 5 , 7 , 4 ]
[11, 4 , 3 , 0 , 10, 9 , 2 , 7 , 1 , 6 , 8 , 5 ]
Order: 12
Structure: [12]


In [136]:
proliferations(Transformation.P,series3,4)

Proliferations of the series using P and a transposition of 4 tone-fractions:
[B , Bb, D , Eb, G , F#, G#, E , F , C , C#, A ]
[Eb, D , F#, G , B , Bb, C , G#, A , E , F , C#]
[G , F#, Bb, B , Eb, D , E , C , C#, G#, A , F ]
Order: 3
Structure: [3, 3, 3, 3]
-------------
Identification of notes and numbers:
0  <---> B
11 <---> Bb
3  <---> D
4  <---> Eb
8  <---> G
7  <---> F#
9  <---> G#
5  <---> E
6  <---> F
1  <---> C
2  <---> C#
10 <---> A


In [137]:
proliferations(Transformation.RI,series7,2)

Proliferations of the series using RI and a transposition of 2 tone-fractions:
[C , G#, G , C#, E , D , A#, D#, B , F , F#, A ]
[F , G#, A , D#, B , E , C , A#, C#, G , F#, D ]
[G , G#, D , A#, C#, B , F , C , D#, A , F#, E ]
[A , G#, E , C , D#, C#, G , F , A#, D , F#, B ]
[D , G#, B , F , A#, D#, A , G , C , E , F#, C#]
[E , G#, C#, G , C , A#, D , A , F , B , F#, D#]
[B , G#, D#, A , F , C , E , D , G , C#, F#, A#]
[C#, G#, A#, D , G , F , B , E , A , D#, F#, C ]
[D#, G#, C , E , A , G , C#, B , D , A#, F#, F ]
[A#, G#, F , B , D , A , D#, C#, E , C , F#, G ]
Order: 10
Structure: [1, 1, 10]
-------------
Identification of notes and numbers:
0  <---> C
8  <---> G#
7  <---> G
1  <---> C#
4  <---> E
2  <---> D
10 <---> A#
3  <---> D#
11 <---> B
5  <---> F
6  <---> F#
9  <---> A


In [138]:
proliferations(Transformation.RI,[0, 6, 8, 5, 7, 11, 4, 3, 9, 10, 1, 2],5)

Proliferations of the series using RI and a transposition of 5 tone-fractions:
[0 , 6 , 8 , 5 , 7 , 11, 4 , 3 , 9 , 10, 1 , 2 ]
[3 , 4 , 7 , 8 , 2 , 1 , 6 , 10, 0 , 9 , 11, 5 ]
[10, 6 , 2 , 7 , 5 , 11, 4 , 9 , 3 , 0 , 1 , 8 ]
[9 , 4 , 5 , 2 , 8 , 1 , 6 , 0 , 10, 3 , 11, 7 ]
Order: 4
Structure: [2, 2, 4, 4]


In [139]:
proliferations(Transformation.RI,[0,5,3,2,6,1,4],0)

Proliferations of the series using RI and a transposition of 0 tone-fractions:
[0, 5, 3, 2, 6, 1, 4]
[3, 6, 1, 5, 4, 2, 0]
[1, 4, 2, 6, 0, 5, 3]
[2, 0, 5, 4, 3, 6, 1]
[5, 3, 6, 0, 1, 4, 2]
[6, 1, 4, 3, 2, 0, 5]
[4, 2, 0, 1, 5, 3, 6]
Order: 7
Structure: [7]


In [140]:
proliferations(Transformation.R,series1,1)

Proliferations of the series using R and a transposition of 1 tone-fractions:
[C , F , E , C#, B , Bb, Eb, Ab, D , G , A , F#]
[G , Bb, Ab, Eb, A , E , B , C , D , F , F#, C#]
[F , E , C , B , F#, Ab, A , G , D , Bb, C#, Eb]
[Bb, Ab, G , A , C#, C , F#, F , D , E , Eb, B ]
[E , C , F , F#, Eb, G , C#, Bb, D , Ab, B , A ]
[Ab, G , Bb, C#, B , F , Eb, E , D , C , A , F#]
[C , F , E , Eb, A , Bb, B , Ab, D , G , F#, C#]
[G , Bb, Ab, B , F#, E , A , C , D , F , C#, Eb]
[F , E , C , A , C#, Ab, F#, G , D , Bb, Eb, B ]
[Bb, Ab, G , F#, Eb, C , C#, F , D , E , B , A ]
[E , C , F , C#, B , G , Eb, Bb, D , Ab, A , F#]
[Ab, G , Bb, Eb, A , F , B , E , D , C , F#, C#]
[C , F , E , B , F#, Bb, A , Ab, D , G , C#, Eb]
[G , Bb, Ab, A , C#, E , F#, C , D , F , Eb, B ]
[F , E , C , F#, Eb, Ab, C#, G , D , Bb, B , A ]
[Bb, Ab, G , C#, B , C , Eb, F , D , E , A , F#]
[E , C , F , Eb, A , G , B , Bb, D , Ab, F#, C#]
[Ab, G , Bb, B , F#, F , A , E , D , C , C#, Eb]
[C , F , E , A , C#, Bb, F#, Ab, D , G ,

In [141]:
proliferations(Transformation.R,series4,2)

Proliferations of the series using R and a transposition of 2 tone-fractions:
[B , C , A , G#, A#, F , F#, G , E , D#, C#, D ]
[E , D#, F , F#, A , G#, G , C , A#, B , D , C#]
[A#, B , G#, G , F , F#, C , D#, A , E , C#, D ]
[A , E , F#, C , G#, G , D#, B , F , A#, D , C#]
[F , A#, G , D#, F#, C , B , E , G#, A , C#, D ]
[G#, A , C , B , G , D#, E , A#, F#, F , D , C#]
[F#, F , D#, E , C , B , A#, A , G , G#, C#, D ]
[G , G#, B , A#, D#, E , A , F , C , F#, D , C#]
[C , F#, E , A , B , A#, F , G#, D#, G , C#, D ]
[D#, G , A#, F , E , A , G#, F#, B , C , D , C#]
Order: 10
Structure: [2, 10]
-------------
Identification of notes and numbers:
0  <---> B
1  <---> C
10 <---> A
9  <---> G#
11 <---> A#
6  <---> F
7  <---> F#
8  <---> G
5  <---> E
4  <---> D#
2  <---> C#
3  <---> D


In [142]:
#Running proliferations_data in a loop for each transformation, n in range(1,max+1) and t in range(0,n)
#gives all this data of proliferating series up to the desired number of notes max.
#Don't forget to change "desiredFolder" to the actual folder you want the files in.

#Currently it will run for all series up to 12 notes, which is quite slow, at least several hours running.

#The completeList folders are occupy several GB for n=12, since they show every possible series, so if you don't want them, just
#comment (put a # in the beggining of the line) the lines that are highlighted with an arrow <------------
#The folder Proliferations_data in this repository has the files that running this program creates, including the cases of
#P and I, and excluding the CompleteList folders.

for n in range(1,10):
    for t in range(0,n):
        proliferations_data(Transformation.RI,t,n,r"desiredFolder")
        proliferations_data(Transformation.R ,t,n,r"desiredFolder")
        proliferations_data(Transformation.P ,t,n,r"desiredFolder")
        proliferations_data(Transformation.I ,t,n,r"desiredFolder")
        #Running it for P and I will give us useless information, since the permutations are essentially independent of the specific series